# Sprint 5 — Graph A Feature Ablation Runner

**Runner-only notebook. Model, preprocessing, evaluation ve plot logic notebook içinde değildir.**
Tüm bilimsel kod `src/`, `scripts/`, `configs/` altındadır.

Execution plan: `docs/exec-plans/active/005-sprint5-epigenetic-ablation.md`  
Runner boundary: `colab/README.md`

---
**Başlamadan önce kontrol et:**
- [ ] Colab runtime GPU seçildi mi?
- [ ] Drive'da `crispr_gnn_offtarget/data/` altında raw dataset veya processed parquet var mı?
- [ ] Branch GitHub'a push edildi mi? (`sprint5/epigenetic-ablation`)
- [ ] Bu run test sonucuna göre feature/hyperparameter değiştirme amacı taşımıyor mu?

## ADIM 1 — Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ADIM 2 — Repo Clone ve Checkout

In [ ]:
%%bash
set -euo pipefail
pip install uv --quiet
git clone https://github.com/YasinEkici/crispr-gnn-offtarget.git crispr-gnn-offtarget
cd crispr-gnn-offtarget
git checkout sprint5/epigenetic-ablation
echo "=== Commit SHA ==="
git rev-parse HEAD

## ADIM 3 — Dependency Sync ve GPU Kontrolü

In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
uv sync
uv run python - <<'PY'
import torch
try:
    import torch_geometric
    pyg_version = torch_geometric.__version__
except Exception as exc:
    pyg_version = f'unavailable: {exc}'
print('torch         :', torch.__version__)
print('pyg           :', pyg_version)
print('cuda_available:', torch.cuda.is_available())
print('cuda_version  :', torch.version.cuda)
print('device        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
PY

## ADIM 4 — Drive Data Kopyala

Bu cell Drive'daki proje data klasörünü Colab lokal diske kopyalar. Büyük dosyalar repo'ya commit edilmez.

In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
DRIVE_ROOT="/content/drive/MyDrive/crispr_gnn_offtarget"
mkdir -p data
if [ -d "$DRIVE_ROOT/data/raw" ]; then
  mkdir -p data/raw
  rsync -a "$DRIVE_ROOT/data/raw/" data/raw/
fi
if [ -d "$DRIVE_ROOT/data/processed" ]; then
  mkdir -p data/processed
  rsync -a "$DRIVE_ROOT/data/processed/" data/processed/
fi
find data -maxdepth 4 -type f | sort | head -50

## ADIM 5 — Sprint 5 Graph A Artefact Üret

In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
uv run python scripts/build_sprint5_graph_a_features.py \
  --data-config configs/data/mak2022.yaml \
  --schema-config configs/sweeps/graph_schema_ablation.yaml \
  --artifact-dir data/processed/graphs/sprint5 \
  --report-path outputs/sprint5/graph_a_feature_ablation_artifact_report.md
uv run python - <<'PY'
from pathlib import Path
from crispr_gnn.graph.graph_schemas import GRAPH_A
from crispr_gnn.graph.pyg_dataset import Sprint3HeteroDataLoader
g = Sprint3HeteroDataLoader(Path('data/processed/graphs/sprint5')).load(GRAPH_A)
print(g.manifest['feature_tables'])
PY

## ADIM 6 — Sprint 5 Ablation Train

Bu cell full run başlatır. Test sonucuna göre feature/hyperparameter değiştirme yapılmaz.

In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
RUN_ID="sprint5_graph_a_feature_ablation_seed42_$(date -u +%Y%m%d)"
uv run python scripts/run_sprint5_feature_ablation.py \
  --config configs/sweeps/sprint5_graph_a_feature_ablation.yaml \
  --run-id "$RUN_ID"
echo "$RUN_ID" > /content/sprint5_run_id.txt

## ADIM 7 — Returned Outputs Drive'a Kopyala

In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
DRIVE_ROOT="/content/drive/MyDrive/crispr_gnn_offtarget"
RUN_BASENAME=$(cat /content/sprint5_run_id.txt)
LATEST_OUT="outputs/sprint5/graph_a_feature_ablation/$RUN_BASENAME"
DRIVE_OUT="$DRIVE_ROOT/returned_outputs/$RUN_BASENAME"
mkdir -p "$DRIVE_ROOT/returned_outputs"
rm -rf "$DRIVE_OUT"
cp -r "$LATEST_OUT" "$DRIVE_OUT"
echo "Copied to: $DRIVE_OUT"
find "$DRIVE_OUT" -maxdepth 3 -type f | sort | head -80

## ADIM 8 — Kısa Sonuç Özeti

In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
RUN_BASENAME=$(cat /content/sprint5_run_id.txt)
LATEST_OUT="outputs/sprint5/graph_a_feature_ablation/$RUN_BASENAME"
RESULTS="$LATEST_OUT/sprint5_graph_a_feature_ablation_results.csv"
uv run python - <<PY
import pandas as pd
results = pd.read_csv('$RESULTS')
cols = ['feature_set','test_auprc','test_auroc','test_f1','test_macro_f1','test_mcc','test_specificity','test_tn','test_fp','test_fn','test_tp']
print(results[cols].to_string(index=False))
PY